In [ ]:
# Import first necessary libraries for data handling, statistics, and plotting.
# Each cell contains also individual libs to comment in if only specific part is needed.
# Some cells also contain optional print commands to check if you are actually getting the correct data and dimensions

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Change file name here, if necessary
df = pd.read_csv('/content/AHOI_Survey_HumanAITeaming_FINISHED_TRUE_values.csv')

# Optional print cmds to check dimensions and first db entries
#print(f"Original DataFrame shape: {df.shape}")
#print("Original DataFrame head:")
#display(df.head())
print(df.columns)

**Correlation analysis between variables "age/sea experience" and the TIA questionnaire. The hypothesis is that older participants tend to be more skeptical, respectively, show less trust in automation processes (here after presentation of the maritime assistant).**

In [ ]:
# Remove the first two rows, which contain metadata
df_cleaned = df.iloc[2:].copy()

# Convert 'age' and 'sea_service' columns to numeric type
df_cleaned['age'] = pd.to_numeric(df_cleaned['age'], errors='coerce')
df_cleaned['sea_service'] = pd.to_numeric(df_cleaned['sea_service'], errors='coerce')

print("DataFrame after removing metadata rows and converting 'age' and 'sea_service' to numeric:")
display(df_cleaned[['age', 'sea_service']].head())
print(f"Shape of the cleaned DataFrame: {df_cleaned.shape}")

Optional print commands to check data.

In [ ]:
print("Unique values in 'age' column:")
print(df_cleaned['age'].unique())

print("\nUnique values in 'sea_service' column:")
print(df_cleaned['sea_service'].unique())

In [ ]:
# Identify TIA columns for Scenario 1 and Scenario 2
tia_scenario1_cols = [col for col in df_cleaned.columns if 'Scenario1_TIA_' in col]
tia_scenario2_cols = [col for col in df_cleaned.columns if 'Scenario2_TIA_' in col]

# Convert TIA columns to numeric (coercing errors to NaN)
df_cleaned[tia_scenario1_cols] = df_cleaned[tia_scenario1_cols].apply(pd.to_numeric, errors='coerce')
df_cleaned[tia_scenario2_cols] = df_cleaned[tia_scenario2_cols].apply(pd.to_numeric, errors='coerce')

# Define items to be reverse-coded
reverse_code_items = ['_6', '_7', '_8', '_9', '_10', '_11']

# Apply reverse-coding to specified TIA items (assuming a 5-point scale: new_value = 6 - old_value)
for col_suffix in reverse_code_items:
    col_s1 = f'Scenario1_TIA{col_suffix}'
    col_s2 = f'Scenario2_TIA{col_suffix}'
    if col_s1 in df_cleaned.columns:
        df_cleaned[col_s1] = 6 - df_cleaned[col_s1]
    if col_s2 in df_cleaned.columns:
        df_cleaned[col_s2] = 6 - df_cleaned[col_s2]

# Calculate average TIA score for each scenario AFTER reverse-coding
df_cleaned['Scenario1_TIA_score'] = df_cleaned[tia_scenario1_cols].mean(axis=1)
df_cleaned['Scenario2_TIA_score'] = df_cleaned[tia_scenario2_cols].mean(axis=1)

print("Calculated Scenario1_TIA_score and Scenario2_TIA_score (with reverse-coding). Head of relevant columns:")
display(df_cleaned[['age', 'sea_service', 'Scenario1_TIA_score', 'Scenario2_TIA_score']].head())

# Calculate correlations
correlation_matrix = df_cleaned[['age', 'sea_service', 'Scenario1_TIA_score', 'Scenario2_TIA_score']].corr()
display(correlation_matrix)

# Visualize the correlation matrix
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Age, Sea Service, and TIA Scores (Reverse-Coded)')
plt.show()

**Age and TIA scores:** After reverse-coding, the correlation between age and Scenario1_TIA_score is now a weak positive (0.08), and between age and Scenario2_TIA_score, it's a weak to moderate positive (0.21). This suggests that as age increases, the TIA score also slightly increases. Since a higher TIA score indicates less intrusiveness (because reverse-coded items are now aligned), this means older participants perceive technology as less intrusive.

**Sea Service and TIA scores:** Similarly, the correlations between sea service and TIA scores are now positive (0.03 for Scenario 1 and 0.23 for Scenario 2). This implies that participants with more sea service also tend to perceive technology as less intrusive.

**TIA scores between scenarios:** The strong positive correlation (0.73) between Scenario1_TIA_score and Scenario2_TIA_score remains, indicating consistent perceptions of technology intrusiveness across both scenarios.

**Conclusion:**
With reverse-coding applied, the quantitative analysis still does not support the hypothesis that "Older participants or participants with higher sea service months are more skeptical towards technology." In fact, the positive correlations, although generally weak, suggest the opposite: older participants and those with more sea service tend to perceive technology as less intrusive or less problematic. This implies they are less skeptical rather than more.

**Correlation analysis between variables "age/sea experience" and sentiment analysis of advantages/disadvantages of using a maritime assistant. The hypothesis is that older participants tend to express more negative sentiment than younger participants.**

In [ ]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

# Download necessary NLTK data for sentiment analysis
nltk.download('vader_lexicon')
nltk.download('wordnet')
nltk.download('punkt')

# Ensure 'advantages' and 'disadvantages' columns are string type and fill NaN with empty strings
df_cleaned['advantages'] = df_cleaned['advantages'].astype(str).fillna('')
df_cleaned['disadvantages'] = df_cleaned['disadvantages'].astype(str).fillna('')

# Function to get sentiment polarity using TextBlob
def get_sentiment_polarity(text):
    return TextBlob(text).sentiment.polarity

# Apply sentiment analysis to 'advantages' and 'disadvantages'
df_cleaned['advantages_sentiment'] = df_cleaned['advantages'].apply(get_sentiment_polarity)
df_cleaned['disadvantages_sentiment'] = df_cleaned['disadvantages'].apply(get_sentiment_polarity)

# Group by 'age' and calculate the mean sentiment scores
sentiment_by_age = df_cleaned.groupby('age')[['advantages_sentiment', 'disadvantages_sentiment']].mean().reset_index()

print("Average sentiment polarity for 'advantages' and 'disadvantages' by age group:")
display(sentiment_by_age)

# Visualize the sentiment by age group
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.barplot(x='age', y='advantages_sentiment', data=sentiment_by_age, palette='viridis')
plt.title('Average Sentiment of "Advantages" by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Average Sentiment Polarity')

plt.subplot(1, 2, 2)
sns.barplot(x='age', y='disadvantages_sentiment', data=sentiment_by_age, palette='magma')
plt.title('Average Sentiment of "Disadvantages" by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Average Sentiment Polarity')

plt.tight_layout()
plt.show()

**Advantages sentiment**: Generally, all age groups express a positive sentiment when discussing 'advantages'. Age group 4 (45-54 years) shows the most positive sentiment, while age group 5 (54 and older) still maintains a positive sentiment, though slightly less than age groups 3 and 4.


**Disadvantages sentiment**: Age groups 1, 2, and 4 show a negative average sentiment, meaning they tend to use more negative words when describing disadvantages. However, age groups 3 and 5 (the oldest group) actually show a slightly positive average sentiment for disadvantages. This suggests that the oldest participants (age group 5) are not using more negative words when describing disadvantages; in fact, their average sentiment is mildly positive, indicating less negativity compared to younger groups.

**Conclusion**: Based on this sentiment analysis, there is no evidence to support the hypothesis that elderly people tend to use more negative words when describing the 'advantages' or 'disadvantages' of technology. In the case of 'disadvantages', the oldest age group (54+) exhibited a slightly positive sentiment, implying a less critical or negative linguistic expression compared to several younger age groups.



**Correlation age/sea experience with ATAS (pre-questionnaire)**

In [ ]:
# Identify ATAS columns
atas_cols = [col for col in df_cleaned.columns if 'ATAS_' in col and col != 'ATAS_score']

# Convert ATAS columns to numeric (coercing errors to NaN)
df_cleaned[atas_cols] = df_cleaned[atas_cols].apply(pd.to_numeric, errors='coerce')

# Calculate average ATAS score
df_cleaned['ATAS_score'] = df_cleaned[atas_cols].mean(axis=1)

print("Calculated ATAS_score. Head of relevant columns:")
display(df_cleaned[['age', 'sea_service', 'ATAS_score']].head())

# Calculate correlations including ATAS_score
correlation_matrix_atas = df_cleaned[['age', 'sea_service', 'ATAS_score']].corr()

print("\nCorrelation Matrix (including ATAS Scores):")
display(correlation_matrix_atas)

# Visualize the correlation matrix
plt.figure(figsize=(6, 5))
sns.heatmap(correlation_matrix_atas, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Age, Sea Service, and ATAS Scores')
plt.show()

**Result:** The correlation analysis reveals very low correlation "age" - "ATAS\_score" = 0.037526. "sea_experience" - "ATAS\_score" = 0.045469

**Conclusions:** Given that the ATAS statements are negated (e.g., "I do NOT..."), a higher ATAS score would imply greater agreement with these negated statements. This could mean slightly less trust or acceptance of automation, but the correlations are so close to zero that they do not provide strong evidence for any significant relationship between age/sea service and ATAS scores. The heatmap visually confirms these very weak correlations.